In [2]:
import pandas as pd
import numpy as np

blr = pd.read_csv("../datasets/raw/bangalore.csv")

blr = blr[
    [
        "location",
        "size",
        "total_sqft",
        "bath",
        "balcony",
        "price",
    ]
].copy()

blr["city"] = "Bangalore"

blr["bhk"] = blr["size"].str.extract(r"(\d+)").astype(float)

blr.rename(
    columns={
        "total_sqft": "sqft",
        "price": "price_lakh"
    },
    inplace=True
)

blr = blr[
    [
        "city",
        "location",
        "sqft",
        "bhk",
        "bath",
        "balcony",
        "price_lakh"
    ]
]

In [3]:
hyd = pd.read_csv("../datasets/raw/hyderabad.csv")

hyd["city"] = "Hyderabad"

hyd["bhk"] = (
    hyd["title"]
    .str.extract(r"(\d+)")
    .astype(float)
)

hyd.rename(
    columns={
        "area_insqft": "sqft",
        "price(L)": "price_lakh"
    },
    inplace=True
)

hyd["bath"] = np.nan
hyd["balcony"] = np.nan

hyd = hyd[
    [
        "city",
        "location",
        "sqft",
        "bhk",
        "bath",
        "balcony",
        "price_lakh"
    ]
]

In [4]:
mum = pd.read_csv("../datasets/raw/mumbai.csv")

mum["price_lakh"] = mum["price"] / 100000

mum.rename(
    columns={
        "locality": "location",
        "area": "sqft",
        "bedroom_num": "bhk",
        "bathroom_num": "bath",
        "balcony_num": "balcony"
    },
    inplace=True
)

mum = mum[
    [
        "city",
        "location",
        "sqft",
        "bhk",
        "bath",
        "balcony",
        "price_lakh"
    ]
]

In [5]:
master = pd.concat(
    [blr, hyd, mum],
    ignore_index=True
)

print(master.shape)
master.head()

(88918, 7)


,city,location,sqft,bhk,bath,balcony,price_lakh
0,Bangalore,Electronic City Phase II,1056,2.0,2.0,1.0,39.07
1,Bangalore,Chikka Tirupathi,2600,4.0,5.0,3.0,120.00
2,Bangalore,Uttarahalli,1440,3.0,2.0,3.0,62.00
3,Bangalore,Lingadheeranahalli,1521,3.0,3.0,1.0,95.00
4,Bangalore,Kothanur,1200,2.0,2.0,1.0,51.00


In [6]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88918 entries, 0 to 88917
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   city        88918 non-null  object 
 1   location    88917 non-null  object 
 2   sqft        88918 non-null  object 
 3   bhk         87285 non-null  float64
 4   bath        85185 non-null  float64
 5   balcony     84649 non-null  float64
 6   price_lakh  88918 non-null  float64
dtypes: float64(4), object(3)
memory usage: 4.7+ MB


In [7]:
master.describe()

,bhk,bath,balcony,price_lakh
count,87285.000000,85185.000000,84649.000000,88918.000000
mean,2.136071,2.177320,0.377571,179.239954
std,1.028404,0.972315,0.810728,308.572357
min,0.000000,1.000000,0.000000,0.320000
25%,1.000000,2.000000,0.000000,60.000000
50%,2.000000,2.000000,0.000000,107.000000
75%,3.000000,3.000000,0.000000,200.000000
max,43.000000,40.000000,15.000000,21474.836470


In [8]:
master["sqft"].sample(20)

11121       1200
39176       1000
77912       1164
56263       1695
9107        1280
9369     2249.81
48257        731
72858        557
7070         400
19222       1800
39503       1700
50807        706
29182        550
79746        732
2854        1313
38187       1200
72673        728
23166        730
79070       1800
37511       1458
Name: sqft, dtype: object

In [9]:
def convert_sqft(x):
    try:
        if "-" in str(x):
            a, b = x.split("-")
            return (float(a) + float(b)) / 2

        return float(x)

    except:
        return None

In [10]:
master["sqft"] = master["sqft"].apply(convert_sqft)

In [11]:
master["sqft"].isnull().sum()

np.int64(46)

In [12]:
master.describe()

,sqft,bhk,bath,balcony,price_lakh
count,88872.000000,87285.000000,85185.000000,84649.000000,88918.000000
mean,1079.801678,2.136071,2.177320,0.377571,179.239954
std,889.236178,1.028404,0.972315,0.810728,308.572357
min,1.000000,0.000000,1.000000,0.000000,0.320000
25%,625.000000,1.000000,2.000000,0.000000,60.000000
50%,900.000000,2.000000,2.000000,0.000000,107.000000
75%,1252.000000,3.000000,3.000000,0.000000,200.000000
max,52272.000000,43.000000,40.000000,15.000000,21474.836470


In [13]:
master = master[
    (master["bhk"] > 0) &
    (master["bhk"] <= 10)
]

master = master[
    (master["bath"] <= 10)
]

master = master[
    (master["sqft"] >= 300)
]

master = master[
    (master["sqft"] <= 10000)
]

In [14]:
master["sqft"].sample(20)

83447     480.0
74924    1000.0
6002     1410.0
66542    1500.0
82575     422.0
82572     690.0
49853     775.0
35399     700.0
68885     691.0
20681     700.0
77425    1350.0
45889     434.0
46888     755.0
27851     560.0
74027     820.0
82960     598.0
1742      912.0
8447     1516.0
66051     401.0
27262     705.0
Name: sqft, dtype: float64

In [15]:
master.isnull().sum()

city            0
location        1
sqft            0
bhk             0
bath            0
balcony       514
price_lakh      0
dtype: int64

In [16]:
master["bhk"].value_counts().sort_index()

bhk
1.0     23453
2.0     35783
3.0     19233
4.0      4418
5.0       839
6.0       360
7.0       136
8.0       100
9.0        53
10.0       16
Name: count, dtype: int64

In [17]:
master["bath"].value_counts().sort_index()

bath
1.0     16444
2.0     45377
3.0     16536
4.0      4138
5.0      1197
6.0       427
7.0       129
8.0        80
9.0        49
10.0       14
Name: count, dtype: int64

In [18]:
master.nlargest(10, "price_lakh")[[
    "city",
    "location",
    "sqft",
    "bhk",
    "bath",
    "price_lakh"
]]

,city,location,sqft,bhk,bath,price_lakh
87899,Mumbai South,Prabhadevi,7398.0,6.0,6.0,9247.00000
42187,Mumbai,Marine Lines,6852.0,4.0,4.0,8895.95595
45216,Mumbai,Colaba,6011.0,4.0,4.0,8546.16165
69912,Mumbai,Marine Lines,6521.0,4.0,4.0,8449.19295
66460,Mumbai,Juhu Scheme,7700.0,6.0,6.0,8000.00000
69957,Mumbai,Churchgate,5621.0,4.0,4.0,7644.91964
42186,Mumbai,Marine Drive,4562.0,4.0,4.0,7491.81884
42185,Mumbai,Juhu,6521.0,4.0,4.0,7418.12959
88909,Western Mumbai,Kandivali,2437.0,4.0,4.0,7067.30000
66970,Mumbai,Malabar Hill,5357.0,4.0,4.0,6767.81952


In [19]:
master["balcony"] = master["balcony"].fillna(0)

master.dropna(inplace=True)

In [20]:
master.shape

(84390, 7)

In [21]:
master.to_csv(
    "../datasets/processed/bricklens_master.csv",
    index=False
)

OSError: Cannot save file into a non-existent directory: '..\datasets\processed'

In [22]:
import os

os.makedirs("../datasets/processed", exist_ok=True)

In [23]:
master["price_per_sqft"] = (
    master["price_lakh"] * 100000
) / master["sqft"]

In [24]:
master["luxury_score"] = (
    master["bhk"] * 2
    + master["bath"]
    + master["balcony"]
)

In [25]:
location_counts = master["location"].value_counts()

master["location_popularity"] = (
    master["location"]
    .map(location_counts)
)

In [26]:
pip install plotly

^C
Note: you may need to restart the kernel to use updated packages.


  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
import os

os.makedirs("../datasets/processed", exist_ok=True)

master.to_csv(
    "../datasets/processed/bricklens_featured.csv",
    index=False
)

print("Dataset saved!")

Dataset saved!


In [28]:
import os

print(os.listdir("../datasets/processed"))

['bricklens_featured.csv']


In [29]:
import plotly.express as px

city_avg = (
    master.groupby("city")["price_lakh"]
    .mean()
    .reset_index()
)

fig = px.bar(
    city_avg,
    x="city",
    y="price_lakh",
    title="Average Property Price by City"
)

fig.show()

In [30]:
master[master["city"] == "Hyderabad"].shape

(0, 10)

In [31]:
print(hyd.shape)
hyd.head()

(3660, 7)


,city,location,sqft,bhk,bath,balcony,price_lakh
0,Hyderabad,Nizampet,1805,3.0,NaN,NaN,108.00
1,Hyderabad,Bachupally,1560,3.0,NaN,NaN,85.80
2,Hyderabad,Dundigal,1070,2.0,NaN,NaN,55.64
3,Hyderabad,Pocharam,1210,2.0,NaN,NaN,60.48
4,Hyderabad,Kollur,1900,3.0,NaN,NaN,113.00


In [32]:
hyd["city"].value_counts()

city
Hyderabad    3660
Name: count, dtype: int64

In [33]:
master["city"].value_counts()

city
Mumbai            70196
Bangalore         13144
Western Mumbai      488
Central Mumbai      360
Mumbai South        202
Name: count, dtype: int64

In [34]:
print("BLR:", blr.shape)
print("HYD:", hyd.shape)
print("MUM:", mum.shape)

BLR: (13320, 7)
HYD: (3660, 7)
MUM: (71938, 7)


In [35]:
master.shape

(84390, 10)

In [36]:
master["city"].unique()

array(['Bangalore', 'Mumbai', 'Mumbai South', 'Central Mumbai',
       'Western Mumbai'], dtype=object)

In [37]:
master["city"].value_counts(dropna=False)

city
Mumbai            70196
Bangalore         13144
Western Mumbai      488
Central Mumbai      360
Mumbai South        202
Name: count, dtype: int64

In [38]:
master["balcony"] = master["balcony"].fillna(0)
master["bath"] = master["bath"].fillna(master["bhk"])

In [39]:
master = pd.concat(
    [blr, hyd, mum],
    ignore_index=True
)

In [40]:
master["balcony"] = master["balcony"].fillna(0)

In [41]:
master["bath"] = master["bath"].fillna(master["bhk"])

In [42]:
master["location"] = master["location"].fillna("Unknown")

In [43]:
master["city"].value_counts()

city
Mumbai            70882
Bangalore         13320
Hyderabad          3660
Western Mumbai      489
Central Mumbai      362
Mumbai South        205
Name: count, dtype: int64

In [44]:
master["city"] = master["city"].replace({
    "Western Mumbai": "Mumbai",
    "Central Mumbai": "Mumbai",
    "Mumbai South": "Mumbai"
})

In [45]:
master["city"].value_counts()

city
Mumbai       71938
Bangalore    13320
Hyderabad     3660
Name: count, dtype: int64

In [46]:
import os

os.makedirs("../datasets/processed", exist_ok=True)

master.to_csv(
    "../datasets/processed/bricklens_featured.csv",
    index=False
)

In [47]:
import os

print(os.listdir("../datasets/processed"))

['bricklens_featured.csv']


In [48]:
import numpy as np

def convert_sqft(x):

    x = str(x)

    # Range values like 2100-2850
    if "-" in x:
        try:
            a, b = x.split("-")
            return (float(a) + float(b)) / 2
        except:
            return np.nan

    # Pure numeric values
    try:
        return float(x)
    except:
        return np.nan

In [49]:
master["sqft"] = master["sqft"].apply(convert_sqft)

In [50]:
master = master.dropna(subset=["sqft"])

In [51]:
master["sqft"] = master["sqft"].astype(float)

In [52]:
master.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88872 entries, 0 to 88917
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   city        88872 non-null  object 
 1   location    88872 non-null  object 
 2   sqft        88872 non-null  float64
 3   bhk         87239 non-null  float64
 4   bath        87239 non-null  float64
 5   balcony     88872 non-null  float64
 6   price_lakh  88872 non-null  float64
dtypes: float64(5), object(2)
memory usage: 5.4+ MB


In [53]:
master.to_csv(
    "../datasets/processed/bricklens_featured.csv",
    index=False
)